In [ ]:
import math
import numpy as np
import filters_exp as exp
from matplotlib import pyplot as plt
import matplotlib as mpl
from numba import njit
from numba_progress import ProgressBar

import metrics
import monte_carlo as mc
import visual_tools as vt
from filters import (
    # Environment helpers
    autocorr_matrix_calc,
    # Environment parameter dtypes
    std_env_parameters, laplace_env_parameters, sys_inversion_gaussian_env_parameters, 
    sys_inversion_laplacian_env_parameters, sys_inversion_general_gaussian_env_parameters,
    sys_inversion_impulsive_gaussian_env_parameters,
    # Algorithm parameter dtypes
    GMVC_parameters, NLMS_parameters, LMLS_parameters, MCC_parameters, RZA_LMS_parameters,
    sKF_parameters, sKF_L_parameters, AWU_sKF_parameters, AWU_sKF_L_parameters, 
    JO_NLMS_parameters, Shin_VSS_NLMS_parameters, NP_VSS_NLMS_parameters, ENP_VSS_NLMS_parameters,
    RVSS_NLMS_parameters, GISV_sKF_L_parameters, cvx_comb_sKF_parameters, cvx_comb_sKF_L_parameters,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, 
    std_gaussian_behavior, laplace_noise_behavior, sys_inversion_gaussian_behavior, 
    sys_inversion_laplacian_behavior, sys_inversion_general_gaussian_behavior,
    sys_inversion_impulsive_gaussian_behavior,
    # algorithms
    NLMS_algorithm, GMVC_algorithm, LMLS_algorithm, MCC_algorithm, RZA_LMS_algorithm,
    sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm, sKF_clip_algorithm, 
    JO_NLMS_algorithm, Shin_VSS_NLMS_algorithm, NP_VSS_NLMS_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    AWU_sKF_algorithm, AWU_sKF_L_algorithm, AWU_sKF_algorithm_v2, AWU_sKF_L_algorithm_v2,
    GISV_sKF_algorithm, AWU_sKF_clip_algorithm, ENP_VSS_NLMS_algorithm, RVSS_NLMS_algorithm,
    GISV_sKF_L_algorithm, cvx_comb_sKF, cvx_comb_sKF_L
 )

# Matplotlib configuration
%matplotlib ipympl
%config InlineBackend.figure_format = 'svg'

# General Parameters

In [ ]:
NR = 256
num_workers = 4
num_chunks = 16

N = int(150e3)
L = 16
ho = np.sinc(np.linspace(0,1,L))
ho = ho/np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3 # Gaussian Noise
scale_v = np.sqrt(1e-3/2) # Laplacian Noise

# AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
#AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

Rxx = autocorr_matrix_calc(AR, 1, M = L)
Rxx = var_x*Rxx/Rxx[0,0]
eigenvalues = np.real(np.linalg.eigvals(Rxx))
Rxx_spread = float(np.max(eigenvalues)/np.min(eigenvalues))

# Metrics
desired_metrics = {
    'MSD': metrics.squared_deviation,
    'MAD': lambda algorithm_output, noise_signal, true_system: metrics.power_deviation(algorithm_output, noise_signal, true_system, 1),
    'EMSE': metrics.excess_squared_error,
    'EMAE': lambda algorithm_output, noise_signal, true_system: metrics.excess_power_error(algorithm_output, noise_signal, true_system, 1),
    'MSE': metrics.squared_error,
}

plotted_metrics = ['MSD', 'var', 'EMSE']
metrics_scales = {
    'MSD': vt.dB_scale,
    'var': vt.var_scale,
    'EMSE': vt.dB_scale,
}

# Algorithm Parameters
mu = 0.1
delta = 1e-3

epsilon_gaussian = 5e-9
epsilon_laplacian_1 = 8e-9
epsilon_laplacian_2 = 2e-8
var_tilde_0 = 1.0 #10000*np.linalg.norm(ho)**2/L

# Time Varying System

## Gaussian noise

In [ ]:
NLMS_Parameters_1 = NLMS_parameters(label="NLMS - mu=1.0", mu=1.0, delta=delta)
NLMS_Parameters_01 = NLMS_parameters(label="NLMS - mu=0.1", mu=0.1, delta=delta)
JO_NLMS_Parameters = JO_NLMS_parameters(label="JO-NLMS", m_0=1e-3, var_w_0=0.0, var_eta=var_v)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF-L - Laplacian", epsilon=epsilon_laplacian_1, b_eta=scale_v, v_tilde_0=var_tilde_0)
sKF_clip_Parameters = sKF_L_parameters(label="sKF-Clip", epsilon=epsilon_laplacian_1, b_eta=scale_v, v_tilde_0=var_tilde_0)
GISV_Parameters = sKF_parameters(label="GISV - Gaussian", epsilon=1e3, var_eta=var_v, v_tilde_0=var_tilde_0)
Shin_VSS_NLMS_Parameters = Shin_VSS_NLMS_parameters(label="Shin-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.01/L, var_e_0=2*var_v/delta)
NP_VSS_NLMS_Parameters = NP_VSS_NLMS_parameters(label="NP-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.05/L, var_e_0=2*var_v, zeta=1e-12)
ENP_VSS_NLMS_Parameters = ENP_VSS_NLMS_parameters(label="ENP-VSS-NLMS", var_eta=var_v, delta=delta, K=2, mu_tau=0.025, var_e_0=2*var_v, zeta=1e-3)
RVSS_NLMS_Parameters = RVSS_NLMS_parameters(label="RVSS-NLMS", epsilon=delta, delta_0=10*var_v/(L*var_x), phi=1-1/L)
GISV_sKF_L_Parameters = GISV_sKF_L_parameters(label="GISV-sKF-L", b_eta=scale_v, v_tilde_0=var_tilde_0, epsilon_min=epsilon_laplacian_1)
cvx_comb_sKF_Parameters = cvx_comb_sKF_parameters(
    label="cvx_comb_sKF", epsilon_1=1.0, epsilon_2=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )
cvx_comb_sKF_L_Parameters = cvx_comb_sKF_L_parameters(
    label="cvx_comb_sKF_L", epsilon_1=1.0, epsilon_2=epsilon_laplacian_1, b_eta=scale_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )

env_parameters = sys_inversion_gaussian_env_parameters(ho=ho, AR=AR, var_v=var_v, var_x=var_x)
environment = sys_inversion_gaussian_behavior

Algorithms = (NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, GISV_sKF_algorithm, GISV_sKF_L_algorithm, cvx_comb_sKF, cvx_comb_sKF_L)
Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, GISV_Parameters, GISV_sKF_L_Parameters, cvx_comb_sKF_Parameters, cvx_comb_sKF_L_Parameters)

tvs_gaussian_MC_measures, tvs_gaussian_task_tree = mc.dask_MC_Simulations(
    N, NR, env_parameters, environment, Algorithms, Alg_Parameters, desired_metrics, h0, num_workers = num_workers, num_chunks = num_chunks
    )

In [ ]:
title = f"Gaussian Noise - Time Varying System - Spread={Rxx_spread:.2f}"
fig = vt.plot_metrics(
    tvs_gaussian_MC_measures,
    plotted_metrics,
    metrics_scales=metrics_scales,
    axis_size=(10, 4),
    title = title
    )

plt.show()
plt.savefig(f"Figures/Gaussian_Noise_Time_Varying_System_Spread_{Rxx_spread:.2f}.pdf")

## Laplacian Noise

In [ ]:
NLMS_Parameters_1 = NLMS_parameters(label="NLMS - mu=1.0", mu=1.0, delta=delta)
NLMS_Parameters_01 = NLMS_parameters(label="NLMS - mu=0.1", mu=0.1, delta=delta)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF-L - Laplacian", epsilon=epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0)
Shin_VSS_NLMS_Parameters = Shin_VSS_NLMS_parameters(label="Shin-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.0065/L, var_e_0=2*var_v/delta)
NP_VSS_NLMS_Parameters = NP_VSS_NLMS_parameters(label="NP-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.02/L, var_e_0=2*var_v, zeta=1e-12)
ENP_VSS_NLMS_Parameters = ENP_VSS_NLMS_parameters(label="ENP-VSS-NLMS", var_eta=var_v, delta=delta, K=2, mu_tau=0.01, var_e_0=2*var_v, zeta=1e-3)
GISV_sKF_L_Parameters = GISV_sKF_L_parameters(label="GISV-sKF-L", b_eta=scale_v, v_tilde_0=var_tilde_0, epsilon_min=epsilon_laplacian_2)
cvx_comb_sKF_Parameters = cvx_comb_sKF_parameters(
    label="cvx_comb_sKF", epsilon_1=1.0, epsilon_2=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )
cvx_comb_sKF_L_Parameters = cvx_comb_sKF_L_parameters(
    label="cvx_comb_sKF_L", epsilon_1=1.0, epsilon_2=epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )

env_parameters = sys_inversion_laplacian_env_parameters(ho=ho, AR=AR, scale_v=scale_v, var_x=var_x)
environment = sys_inversion_laplacian_behavior

Algorithms = (NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, GISV_sKF_algorithm, GISV_sKF_L_algorithm, cvx_comb_sKF, cvx_comb_sKF_L)
Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, GISV_Parameters, GISV_sKF_L_Parameters, cvx_comb_sKF_Parameters, cvx_comb_sKF_L_Parameters)

tvs_laplacian_MC_measures, tvs_laplacian_task_tree = mc.dask_MC_Simulations(
    N, NR, env_parameters, environment, Algorithms, Alg_Parameters, desired_metrics, h0, num_workers = num_workers, num_chunks = num_chunks
    )

In [ ]:
title = f"Laplacian Noise - Time Varying System - Spread={Rxx_spread:.2f}"
fig = vt.plot_metrics(
    tvs_laplacian_MC_measures,
    plotted_metrics,
    metrics_scales=metrics_scales,
    axis_size=(10, 4),
    title = title
    )

plt.show()
plt.savefig(f"Figures/Laplacian_Noise_Time_Varying_System_Spread_{Rxx_spread:.2f}.pdf")

## Heavy tail noise with generalized Gaussian $(\beta^* = 0.2)$

In [ ]:
# Noise parameters
var_eta = var_v
beta = 0.2
alpha = np.sqrt(var_eta*math.gamma(1/beta)/math.gamma(3/beta))

NLMS_Parameters_1 = NLMS_parameters(label="NLMS - mu=1.0", mu=1.0, delta=delta)
NLMS_Parameters_01 = NLMS_parameters(label="NLMS - mu=0.1", mu=0.1, delta=delta)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
Shin_VSS_NLMS_Parameters = Shin_VSS_NLMS_parameters(label="Shin-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.0065/L, var_e_0=2*var_v/delta)
NP_VSS_NLMS_Parameters = NP_VSS_NLMS_parameters(label="NP-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.02/L, var_e_0=2*var_v, zeta=1e-12)
ENP_VSS_NLMS_Parameters = ENP_VSS_NLMS_parameters(label="ENP-VSS-NLMS", var_eta=var_v, delta=delta, K=2, mu_tau=0.01, var_e_0=2*var_v, zeta=1e-3)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF-L - Laplacian", epsilon=40*epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0)
GISV_sKF_L_Parameters = GISV_sKF_L_parameters(label="GISV-sKF-L", b_eta=scale_v, v_tilde_0=0.0, epsilon_min=40*epsilon_laplacian_2)
GISV_Parameters = sKF_parameters(label="GISV - Gaussian", epsilon=1e3, var_eta=var_v, v_tilde_0=var_tilde_0)
cvx_comb_sKF_Parameters = cvx_comb_sKF_parameters(
    label="cvx_comb_sKF", epsilon_1=1.0, epsilon_2=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )
cvx_comb_sKF_L_Parameters = cvx_comb_sKF_L_parameters(
    label="cvx_comb_sKF_L", epsilon_1=1.0, epsilon_2=40*epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )

env_parameters = sys_inversion_general_gaussian_env_parameters(ho=ho, AR=AR, beta=beta, scale_v=alpha, var_x=var_x)
environment = sys_inversion_general_gaussian_behavior

Algorithms = (NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, GISV_sKF_algorithm, GISV_sKF_L_algorithm, cvx_comb_sKF, cvx_comb_sKF_L)
Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, GISV_Parameters, GISV_sKF_L_Parameters, cvx_comb_sKF_Parameters, cvx_comb_sKF_L_Parameters)

tvs_general_gaussian_MC_measures, tvs_general_gaussian_task_tree = mc.dask_MC_Simulations(
    N, NR, env_parameters, environment, Algorithms, Alg_Parameters, desired_metrics, h0, num_workers = num_workers, num_chunks = num_chunks
    )

In [ ]:
title = f"General Gaussian Noise $({r'beta^*'}={env_parameters.beta:.2f})$ - Time Varying System - Spread={Rxx_spread:.2f}"
fig = vt.plot_metrics(
    tvs_general_gaussian_MC_measures,
    plotted_metrics,
    metrics_scales=metrics_scales,
    axis_size=(10, 4),
    title = title
    )

plt.show()
plt.savefig(f"Figures/General_Gaussian_Noise_beta_star_{env_parameters.beta:.2f}_Time_Varying_System_Spread_{Rxx_spread:.2f}.pdf")

## Impulsive Gaussian

In [ ]:
NLMS_Parameters_1 = NLMS_parameters(label="NLMS - mu=1.0", mu=1.0, delta=delta)
NLMS_Parameters_01 = NLMS_parameters(label="NLMS - mu=0.1", mu=0.1, delta=delta)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
Shin_VSS_NLMS_Parameters = Shin_VSS_NLMS_parameters(label="Shin-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.0065/L, var_e_0=2*var_v/delta)
NP_VSS_NLMS_Parameters = NP_VSS_NLMS_parameters(label="NP-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.02/L, var_e_0=2*var_v, zeta=1e-12)
ENP_VSS_NLMS_Parameters = ENP_VSS_NLMS_parameters(label="ENP-VSS-NLMS", var_eta=var_v, delta=delta, K=2, mu_tau=0.01, var_e_0=2*var_v, zeta=1e-3)
GISV_Parameters = sKF_parameters(label="GISV - Gaussian", epsilon=1e3, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF-L - Laplacian", epsilon=40*epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0)
GISV_sKF_L_Parameters = GISV_sKF_L_parameters(label="GISV-sKF-L", b_eta=scale_v, v_tilde_0=0.0, epsilon_min=40*epsilon_laplacian_2)
cvx_comb_sKF_Parameters = cvx_comb_sKF_parameters(
    label="cvx_comb_sKF", epsilon_1=1.0, epsilon_2=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )
cvx_comb_sKF_L_Parameters = cvx_comb_sKF_L_parameters(
    label="cvx_comb_sKF_L", epsilon_1=1.0, epsilon_2=40*epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0, mu_a=1.0, a_plus=4.0, tau=1-10/L, l=1-1/L, reg_rho=1e-2, lambda_0=0.7
    )

p = 0.9
c_1 = 0.01
c_2 = (1-p*c_1)/(1-p)
env_parameters = sys_inversion_impulsive_gaussian_env_parameters(ho=ho, AR=AR, p=p, var_1=c_1*var_v, var_2=c_2*var_v, var_x=var_x)
environment = sys_inversion_impulsive_gaussian_behavior

Algorithms = (NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, GISV_sKF_algorithm, GISV_sKF_L_algorithm, cvx_comb_sKF, cvx_comb_sKF_L)
Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, GISV_Parameters, GISV_sKF_L_Parameters, cvx_comb_sKF_Parameters, cvx_comb_sKF_L_Parameters)

tvs_impulsive_gaussian_MC_measures, tvs_impulsive_gaussian_task_tree = mc.dask_MC_Simulations(
    N, NR, env_parameters, environment, Algorithms, Alg_Parameters, desired_metrics, h0, num_workers = num_workers, num_chunks = num_chunks
    )

In [ ]:
title = f"Time-Varying System - Impulsive Gaussian Environment - Spread={Rxx_spread:.2f}"
fig = vt.plot_metrics(
    tvs_impulsive_gaussian_MC_measures,
    plotted_metrics,
    metrics_scales=metrics_scales,
    axis_size=(10, 4),
    title = title
    )

plt.show()
plt.savefig(f"Figures/Impulsive_Gaussian_Noise_Time_Varying_System_Spread_{Rxx_spread:.2f}.pdf")